# GitHub repositories and users recommendations by embeddings

## Problem Statement

Currently, GitHub has two possibilities to explore users and repositories:
1. Direct search by search term leveraging names and tags.
2. Recommender system under 'Explore' tab which gives suggestions to a user based on his usage of service.

However, there is no possibility to perform a search of connected entities. E.g., find repositories or users highly related to each other.

## Goal of the Project

The goal of this project is to build GitHub repository search/recommender system, which would allow exploring connected repositories and people, by leveraging the underlying graph structure of the repositories database.

## Implemented ML solution

It was decided to build graph nodes embeddings (`repo2vec` and `user2vec`) for the entire GitHub database using [PyTorch-BigGraph (PBG)](https://github.com/facebookresearch/PyTorch-BigGraph). On top of the embeddings representation, we have built query tool with the ranking engine.

# Pipeline

In [1]:
from resources.utils import *

# DUMP_DATE = '2019-06-01'
DUMP_DATE = '2020-07-17'
DUMP_DATE

'2020-07-17'

## Dataset: preprocessing

The goal of this stage is to process the GitHub dataset from http://ghtorrent.org/ project into directed graph. <br>
The source data consists from .csv files which represent tabular SQL data. Each file is one table in source database. <br>
Total size of the dataset is 100GB. The size of the tables used for graph building is ~7GB.

*Database schema:*
![image.png](http://ghtorrent.org/files/schema.png)

The following tables are used for the graph representation:
* Followers
* Watchers
* Project Members

##### Graph structure 
Following nodes are defined as points of interest for this project:
* Users
* Projects

Types of graph edges:
* Follows
* Watches
* Is member of

Possible edges:
* User => follow => User
* User => follow => Project
* User => watch => User
* User => watch => Project
* User => is member => Project

### Download data (http://ghtorrent.org/)

Run `db_download.sh` script (at terminal) to download dump

Extract relationships tables (CSV): `followers.csv`, `watchers.csv`, `project_members.csv`

and metadata: `users.csv`, `projects.csv`

to `data/mysql-%Y-%m-%d` folder (or change directories code dependencies at notebooks)

In [29]:
# !!! not working !!! to run through CLI
# !./db_download.sh {DUMP_DATE}

http://ghtorrent-downloads.ewi.tudelft.nl/mysql/mysql-2020-07-17.tar.gz


In [38]:
from time import sleep

DOWNLOAD_LOG_FILE = f"./logs/downloads_{DUMP_DATE}.out"

prev_line = !tail -n 2 {DOWNLOAD_LOG_FILE} | head -n 1
prev_line = prev_line[0]
for i in tn(range(60*4)):
    curr_line = !tail -n 2 {DOWNLOAD_LOG_FILE} | head -n 1
    curr_line = curr_line[0]
    if curr_line == prev_line:
        break
    print(pd.datetime.now(), curr_line)
    prev_line = curr_line
    sleep(60)

In [39]:
!tail {DOWNLOAD_LOG_FILE}

134118900K .......... .......... .......... .......... .......... 99% 50.7M 0s
134118950K .......... .......... .......... .......... .......... 99% 60.4M 0s
134119000K .......... .......... .......... .......... .......... 99% 37.2M 0s
134119050K .......... .......... .......... .......... .......... 99% 87.0M 0s
134119100K .......... .......... .......... .......... .......... 99% 62.7M 0s
134119150K .......... .......... .......... .......... .......... 99% 15.3M 0s
134119200K ..                                                    100% 15.7M=3m9s

2020-10-02 12:14:38 (48.1 MB/s) - ‘data/mysql-2020-07-17.tar.gz’ saved [137338062980/137338062980]



In [40]:
!head ./data/

ID	URL	Username	Name	Readme	Topics	Title	Starred	Branch	LastUpdate
1723826	https://github.com/coallaoh/AIP	coallaoh	AIP	"Adversarial Image Perturbation for Privacy Protection &ndash; A Game Theory Perspective, ICCV&rsquo;17
Seong Joon Oh, Mario Fritz, Bernt Schiele.
Max-Planck Institute for Informatics.
, ICCV&rsquo;17
Users like sharing personal photos with others through social media. At the same time, they might want to make automatic identification in such photos difficult or even impossible. Classic obfuscation methods such as blurring are not only unpleasant but also not as effective as one would expect. Recent studies on adversarial image perturbations (AIP) suggest that it is possible to confuse recognition systems effectively without unpleasant artifacts. However, in the presence of counter measures against AIPs, it is unclear how effective AIP would be in particular when the choice of counter measure is unknown. Game theory provides tools for studying the interaction between 

In [42]:
!tar --help

Usage: tar [OPTION...] [FILE]...
GNU 'tar' saves many files together into a single tape or disk archive, and can
restore individual files from the archive.

Examples:
  tar -cf archive.tar foo bar  # Create archive.tar from files foo and bar.
  tar -tvf archive.tar         # List all files in archive.tar verbosely.
  tar -xf archive.tar          # Extract all files from archive.tar.

 Local file name selection:

      --add-file=FILE        add given FILE to the archive (useful if its name
                             starts with a dash)
  -C, --directory=DIR        change to directory DIR
      --exclude=PATTERN      exclude files, given as a PATTERN
      --exclude-backups      exclude backup and lock files
      --exclude-caches       exclude contents of directories containing
                             CACHEDIR.TAG, except for the tag file itself
      --exclude-caches-all   exclude directories containing CACHEDIR.TAG
      --exclude-caches-under exclude everything under director

In [43]:
# DUMP_PREFIX = f'mysql-{DUMP_DATE}'
DUMP_PREFIX = 'dump'

!tar -C data/ -xvzf data/mysql-{DUMP_DATE}.tar.gz \
    {DUMP_PREFIX}/project_members.csv \
    {DUMP_PREFIX}/followers.csv \
    {DUMP_PREFIX}/watchers.csv \
    {DUMP_PREFIX}/projects.csv \
    {DUMP_PREFIX}/users.csv

dump/project_members.csv
dump/followers.csv
dump/watchers.csv


In [ ]:
# DUMP_PREFIX = f'mysql-{DUMP_DATE}'
DUMP_PREFIX = 'dump'

!tar -C data/ -xvzf data/mysql-{DUMP_DATE}.tar.gz \
#     {DUMP_PREFIX}/project_members.csv \
#     {DUMP_PREFIX}/followers.csv \
#     {DUMP_PREFIX}/watchers.csv \
    {DUMP_PREFIX}/projects.csv \
    {DUMP_PREFIX}/users.csv

dump/project_members.csv
dump/followers.csv
dump/watchers.csv


### Process SQL structured data into edges

In [48]:
from pyspark import SparkContext, SparkConf, SQLContext

# constants
DATA_FOLDER = f"./data/{DUMP_PREFIX}/"
FOLLOWERS_PATH = DATA_FOLDER + "followers.csv"
WATCHERS_PATH = DATA_FOLDER + "watchers.csv"
PROJECT_MEMBERS_PATH = DATA_FOLDER + "project_members.csv"

# relations constants
IS_MEMBER_OF = "is_member_of"
FOLLOWS = "follows"
WATCHES = "watches"

In [45]:
def addIdentifiers(row, id_1, id_2):
    '''
    Adds identifier for each id to distinguish them
    '''
    row[0] = id_1 + row[0]
    row[1] = id_2 + row[1]
    return row

def swapColumns(row):
    temp = row[0]
    row[0] = row[1]
    row[1] = temp
    return row

def addRelation(row, relation):
    temp = row[1]
    row[1] = relation
    row.append(temp)
    return row

In [46]:
conf = SparkConf()
sc = SparkContext(conf=conf)

Processing of members table

In [49]:
members_rdd = sc.textFile(PROJECT_MEMBERS_PATH).map(lambda x: x.split(",")[:-2])

members_rdd = members_rdd.map(lambda x: addIdentifiers(x, "repo_id_", "user_id_")).map(swapColumns).map(lambda x: addRelation(x,IS_MEMBER_OF))
members_rdd.take(5)

[['user_id_1', 'is_member_of', 'repo_id_1'],
 ['user_id_2', 'is_member_of', 'repo_id_1'],
 ['user_id_4', 'is_member_of', 'repo_id_1'],
 ['user_id_24', 'is_member_of', 'repo_id_3'],
 ['user_id_5465', 'is_member_of', 'repo_id_3']]

Processing of followers table

In [50]:
followers_rdd = sc.textFile(FOLLOWERS_PATH).map(lambda x: x.split(",")[:-1]) \
    .map(lambda x: addIdentifiers(x, "user_id_", "user_id_")) \
    .map(swapColumns) \
    .map(lambda x: addRelation(x, FOLLOWS))

followers_rdd.take(5)

[['user_id_2', 'follows', 'user_id_1'],
 ['user_id_4', 'follows', 'user_id_1'],
 ['user_id_17896', 'follows', 'user_id_1'],
 ['user_id_21523', 'follows', 'user_id_1'],
 ['user_id_29121', 'follows', 'user_id_1']]

Processing of watchers table

In [51]:
watchers_rdd = sc.textFile(WATCHERS_PATH).map(lambda x: x.split(",")[:-1]) \
    .map(lambda x: addIdentifiers(x, "repo_id_", "user_id_")) \
    .map(swapColumns) \
    .map(lambda x: addRelation(x, WATCHES))

watchers_rdd.take(5)

[['user_id_1', 'watches', 'repo_id_1'],
 ['user_id_2', 'watches', 'repo_id_1'],
 ['user_id_4', 'watches', 'repo_id_1'],
 ['user_id_6', 'watches', 'repo_id_1'],
 ['user_id_7', 'watches', 'repo_id_1']]

### Split into train, validation and test datasets

Merge into one entity

In [52]:
rdd = members_rdd.union(followers_rdd).union(watchers_rdd)

sqlContext = SQLContext(sc)
df = sqlContext.createDataFrame(rdd, ['from_entity_id', 'relation', 'to_entity_id'])
df.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges')
shuffledDF = df.randomSplit([0.1, 0.1, 0.8])
val_set = shuffledDF[0]
test_set = shuffledDF[1]
train_set = shuffledDF[2]

Save into files

In [73]:
val_set.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges/validation')
test_set.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges/test')
train_set.coalesce(1).write.option("delimiter", "\t").format('com.databricks.spark.csv').options(header='true').save(DATA_FOLDER + 'graph_edges/train')

AnalysisException: 'path file:/home/jovyan/work/ucu/github-recommendations/data/dump/graph_edges/validation already exists.;'

### Store metadata

Metainfo we store in MongoDB

In [ ]:
def convert_dt(sdt):
    try:
        return pd.datetime.strptime(sdt, '%Y-%m-%d %H:%M:%S')
    except Exception as e:
        print(e)

#### Users

In [5]:
users_df = pd.read_csv(f'{DATA_FOLDER}users.csv',
                       header=None,
                       names=['id', 'login', 'company', 'created_at', 'type', 'fake', 'deleted', 'long', 'lat',
                              'country_code', 'state', 'city', 'location'],
#                        parse_dates=[3],
#                        quoting=1,
#                        quotechar='"',
#                        encoding='utf-8',
                       low_memory=False,
                       error_bad_lines=False).replace({'\\N': None}).set_index('id')
print(users_df.shape)
users_df['created_at'] = users_df['created_at'].map(convert_dt)
users_df = users_df[~users_df['created_at'].isna()]
users_df['fake'] = users_df['fake'].replace({1: True, 0: False, '1': True, '0': False}).astype(bool)
users_df['deleted'] = users_df['deleted'].replace({1: True, 0: False, '1': True, '0': False}).astype(bool)
users_df['long'] = users_df['long'].astype(float)
users_df['lat'] = users_df['lat'].astype(float)
users_df.shape

(30600249, 12)

In [3]:
mc['github']['users'].create_index('login')

'login_1'

In [6]:
to_insert = []

for uid, row in tn(users_df.iterrows(), total=users_df.shape[0]):
    to_insert.append({'_id': uid, **row.to_dict()})
    if len(to_insert) >= 10_000:
        mc['github']['users'].insert_many(to_insert)
        to_insert = []
if len(to_insert):
    mc['github']['users'].insert_many(to_insert)
mc['github']['users'].estimated_document_count()

30600249

Users example
![Users](./resources/images/Users.png)

#### Projects (repos)

In [ ]:
for field in ['name', 'owner_id', 'url']:
    mc['github']['projects'].create_index(field)

In [7]:
try:
    stop = False
    skip_count = 0
    for i in range(1000):
        print(i)
        repos_df = pd.read_csv(f'{DATA_FOLDER}projects.csv',
                               skiprows=10_000_000*i,
                               nrows=10_000_000,
                               header=None,
                               na_values=['\\N'],
                               names=["id", "url", "owner_id", "name", "description", "language",
                                      "created_at", "forked_from", "deleted", "updated_at", "?"],
                               error_bad_lines=False,
                               low_memory=False,
                              ).set_index('id')
        c = repos_df.shape[0]
        if c < 10_000_000:
            stop = True
        repos_df['created_at'] = repos_df['created_at'].map(convert_dt)
        repos_df = repos_df[~repos_df['created_at'].isna()]
        repos_df['updated_at'] = repos_df['updated_at'].map(convert_dt)
        repos_df = repos_df[~repos_df['updated_at'].isna()]
        skip_count += (c - repos_df.shape[0])
        repos_df.index = repos_df.index.astype(int)
        repos_df['url'] = repos_df['url'].astype(str).map(lambda x: x[29:])
        repos_df['deleted'] = repos_df['deleted'].replace({1: True, 0: False, 1: True, 0: False}).astype(bool)
        repos_df['owner_id'] = repos_df['owner_id'].astype(int)
        repos_df['forked_from'] = repos_df['forked_from'].astype(float)
        repos_df['?'] = repos_df['?'].astype(float)

        to_insert = []

        for rid, row in tn(repos_df.iterrows(), total=repos_df.shape[0]):
            to_insert.append({'_id': rid, **row.to_dict()})
            if len(to_insert) >= 10_000:
                mc['github']['projects'].insert_many(to_insert)
                to_insert = []
        if len(to_insert):
            mc['github']['project'].insert_many(to_insert)
        if stop:
            break
    m = f"[projects]: stored {mc['github']['projects'].estimated_document_count()} projects, {skip_count} skiped"
    tgn(m)
except Exception as e:
    m = f'[projects]: error - {e}'
    tgn(m)
m

'[projects]: stored 116010000 projects, 135185 skiped'

Repos example
![Projects](./resources/images/Projects.png)

## Training

![Partitions](./resources/images/Partitions.png)

*Figure 1.* The PBG partitioning scheme for large graphs. **Left:** nodes are divided into $P$ partitions that are sized to fit in memory. Edges are divided into buckets based on the partition of their source and destination nodes. In distributed mode, multiple buckets with non-overlapping partitions can be executed in parallel (red squares). **Center:** Entity types with small cardinality do not have to be partitioned; if all entity types used for tail nodes are unpartitioned, then edges can be divided into $P$ buckets based only on source node partitions. **Right:** the ‘inside-out’ bucket order guarantees that buckets have at least one previously-trained embedding partition. Empirically, this ordering produces better embeddings than other alternatives (or random)

![Architecture](./resources/images/Architecture.png)

*Figure 2.* Memory-efficient batched negative sampling. Embeddings are fetched for the $B$ source and destination entities in a batch of edges, as well as $B$ uniformly-sampled source and destination entities. Each chunk of $B_n/2$ edges is corrupted with all source or destination entities in its chunk, as well as the corresponding chunk of the uniform embeddings, resulting in $B_n$ negative examples per positive edge. The negative scores are computed via a batch matrix multiply.

Lerer, Adam, et al. "PyTorch-BigGraph: A Large-scale Graph Embedding System." arXiv preprint arXiv:1903.12287 (2019).
https://arxiv.org/pdf/1903.12287.pdf

Clear dir from possible previous training (or archive configs)

In [56]:
for TEMPLATE in ['entity_count_all_*.txt', 'graph-*_partitioned/*', 'dictionary.json', 'dynamic_rel_count.txt']:
    !rm -f {DATA_FOLDER}graph_edges/{TEMPLATE}

See `graph_config.py` config file for details (`torchbiggraph_train -h` for help)

Let's exclude 'long tail' where entity (repo/user) has less than {ENTITY_MIN_COUNT} relations

In [57]:
# ENTITY_MIN_COUNT = 30
ENTITY_MIN_COUNT = 2

Creating needed partitions

In [ ]:
%time !torchbiggraph_import_from_tsv --lhs-col=0 --rel-col=1 --rhs-col=2 --entity-min-count={ENTITY_MIN_COUNT} graph_config.py {DATA_FOLDER}graph_edges/graph-*.csv > logs/formating.log

In [70]:
!torchbiggraph_import_from_tsv --help

usage: torchbiggraph_import_from_tsv [-h] -l LHS_COL -r RHS_COL
                                     [--rel-col REL_COL]
                                     [--relation-type-min-count RELATION_TYPE_MIN_COUNT]
                                     [--entity-min-count ENTITY_MIN_COUNT]
                                     config [edge_paths [edge_paths ...]]

positional arguments:
  config                Path to config file
  edge_paths            Input file paths

optional arguments:
  -h, --help            show this help message and exit
  -l LHS_COL, --lhs-col LHS_COL
                        Column index for source entity
  -r RHS_COL, --rhs-col RHS_COL
                        Column index for target entity
  --rel-col REL_COL     Column index for relation entity
  --relation-type-min-count RELATION_TYPE_MIN_COUNT
                        Min count for relation types
  --entity-min-count ENTITY_MIN_COUNT
                        Min count for entities

Config parameters:

entity:

  num

In [60]:
%time !torchbiggraph_import_from_tsv --lhs-col=0 --rel-col=1 --rhs-col=2 --entity-min-count={ENTITY_MIN_COUNT} graph_config.py {DATA_FOLDER}graph_edges/part-*.csv > logs/formating.log

CPU times: user 11.1 s, sys: 2.15 s, total: 13.2 s
Wall time: 19min 28s


In [72]:
%time !torchbiggraph_import_from_tsv --lhs-col=0 --rel-col=1 --rhs-col=2 --entity-min-count={ENTITY_MIN_COUNT} graph_config.py {DATA_FOLDER}graph_edges/part-*.csv > logs/formating.log

CPU times: user 10.8 s, sys: 2.31 s, total: 13.1 s
Wall time: 19min 28s


In [21]:
!head -n 20 logs/formating.log

Looking up relation types in the edge files...
- Found 4 relation types
- Removing the ones with fewer than 1 occurrences...
- Left with 4 relation types
- Shuffling them...
Searching for the entities in the edge files...
Entity type all:
- Found 27406591 entities
- Removing the ones with fewer than 10 occurrences...
- Left with 3366929 entities
- Shuffling them...
Preparing entity path ./data/mysql-2019-06-01/graph_edges:
- Writing count of entity type all and partition 0
- Writing count of entity type all and partition 1
- Writing count of entity type all and partition 2
- Writing count of entity type all and partition 3
- Writing count of dynamic relations
Preparing edge path ./data/mysql-2019-06-01/graph_edges/graph-test_partitioned, out of the edges found in ./data/mysql-2019-06-01/graph_edges/graph-test.csv
- Edges will be partitioned in 4 x 4 buckets.
- Processed 100000 edges so far...


In [74]:
!head -n 20 logs/formating.log

Looking up relation types in the edge files...
- Found 4 relation types
- Removing the ones with fewer than 1 occurrences...
- Left with 4 relation types
- Shuffling them...
Searching for the entities in the edge files...
Entity type all:
- Found 32268021 entities
- Removing the ones with fewer than 2 occurrences...
- Left with 15038812 entities
- Shuffling them...
Preparing entity path ./data/dump/graph_edges:
- Writing count of entity type all and partition 0
- Writing count of entity type all and partition 1
- Writing count of entity type all and partition 2
- Writing count of entity type all and partition 3
- Writing count of dynamic relations
Preparing edge path ./data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000_partitioned, out of the edges found in ./data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000.csv
- Edges will be partitioned in 4 x 4 buckets.
- Processed 100000 edges so far...


In [77]:
for TEMPLATE in ['checkpoint_version.txt', 'embeddings_all_*.h5', 'model.v*.h5']:
    !rm -f {DATA_FOLDER}graph_edges/{TEMPLATE}

Actually, training

In [ ]:
%time !torchbiggraph_train graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/graph-train_partitioned > logs/train.log

In [79]:
%time !torchbiggraph_train graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000_partitioned > logs/train.log

still in queue: 15
still in queue: 14
still in queue: 13
still in queue: 12
still in queue: 11
still in queue: 10
still in queue: 9
still in queue: 8
still in queue: 7
still in queue: 6
still in queue: 5
still in queue: 4
still in queue: 3
still in queue: 2
still in queue: 1
still in queue: 0
still in queue: 15
still in queue: 14
still in queue: 13
still in queue: 12
still in queue: 11
still in queue: 10
still in queue: 9
still in queue: 8
still in queue: 7
still in queue: 6
still in queue: 5
still in queue: 4
still in queue: 3
still in queue: 2
still in queue: 1
still in queue: 0
still in queue: 15
still in queue: 14
still in queue: 13
still in queue: 12
still in queue: 11
still in queue: 10
still in queue: 9
still in queue: 8
still in queue: 7
still in queue: 6
still in queue: 5
still in queue: 4
still in queue: 3
still in queue: 2
still in queue: 1
still in queue: 0
still in queue: 15
still in queue: 14
still in queue: 13
still in queue: 12
still in queue: 11
still in queue: 10
stil

In [80]:
!head logs/train.log
print('-'*100)
!tail logs/train.log

2020-10-03 13:24:51  Loading entity counts...
2020-10-03 13:24:51  Creating workers...
2020-10-03 13:24:51  Initializing global model...
2020-10-03 13:24:51  Starting epoch 1 / 10 edge path 1 / 1 edge chunk 1 / 1
2020-10-03 13:24:51  edge_path= ./data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000_partitioned
2020-10-03 13:24:51  Swapping partitioned embeddings None ( 3 , 3 )
2020-10-03 13:24:51  Loading entities
2020-10-03 13:26:44  ( 3 , 3 ): bucket 1 / 16 : Processed 12504772 edges in 112.06 s ( 0.11 M/sec ); io: 1.54 s ( 1167.88 MB/sec )
2020-10-03 13:26:44  ( 3 , 3 ): loss:  7.19049 , violators_lhs:  24.2728 , violators_rhs:  23.687 , count:  12504772
2020-10-03 13:26:44  Swapping partitioned embeddings ( 3 , 3 ) ( 2 , 2 )
----------------------------------------------------------------------------------------------------
2020-10-03 18:14:59  ( 1 , 0 ): bucket 16 / 16 : Processed 12872060 edges in 104.97 s ( 0.12 M/sec ); io: 0.80 s ( 384.52 MB/sec )
2020-10

## Evaluating

### BigGraph Evaluation

Offline evaluation in the BigGraph performed using `torchbiggraph_eval` that samples some number of negative edges with a particular positive edge. In practice, since we are using sparse graphs - this implementation assumes that randomly given edge is a negative one with high probability. Therefore some positive edges may be mixed with a negative one.

Several metrics are built-in into the library and calculated for each batch separately, and afterwards globally, such as:
 - *Mean Rank (pos_rank)* - the average of the ranks of all positives (lower is better, best is 1).
 - *Mean Reciprocal Rank (MRR)* - the average of the reciprocal of the ranks of all positives (higher is better, best is 1).
 $$\dfrac{1}{n} \sum_{i=1}^{n} \dfrac{1}{rank}, $$
 where $n$ is the number of all positives, and $rank$ is rank of a positive result in a sample
 - *r1* - the fraction of positives that rank better than all their negatives, i.e., have a rank of 1 (higher is better, best is 1).
 - *r10* - the fraction of positives that rank in the top 10 among their negatives (higher is better, best is 1).
 - *r50* - the fraction of positives that rank in the top 50 among their negatives (higher is better, best is 1).
 - *AUC* - an estimation of the probability that a randomly chosen positive scores higher than a randomly chosen negative (any negative, not only the negatives constructed by corrupting that positive).

**Evaluating on train**

In [3]:
!torchbiggraph_eval graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/graph-train_partitioned > logs/eval_train.log

In [82]:
!torchbiggraph_eval graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000_partitioned > logs/eval_train.log

In [4]:
!head logs/eval_train.log
print('-'*100)
!tail logs/eval_train.log

2019-08-14 18:09:29  Starting edge path 1 / 1 (./data/mysql-2019-06-01/graph_edges/graph-train_partitioned)
2019-08-14 18:09:42  ( 0 , 0 ): Processed 6383098 edges in 12 s (0.52M/sec); load time: 0.18 s
2019-08-14 18:09:42  Stats for edge path 1 / 1, bucket ( 0 , 0 ): pos_rank:  6.76915 , mrr:  0.480395 , r1:  0.321494 , r10:  0.813052 , r50:  0.991382 , auc:  0.967479 , count:  6383098
2019-08-14 18:09:54  ( 0 , 1 ): Processed 6584603 edges in 12 s (0.53M/sec); load time: 0.15 s
2019-08-14 18:09:54  Stats for edge path 1 / 1, bucket ( 0 , 1 ): pos_rank:  6.66269 , mrr:  0.48533 , r1:  0.327922 , r10:  0.81517 , r50:  0.992204 , auc:  0.968301 , count:  6584603
2019-08-14 18:10:07  ( 0 , 2 ): Processed 6469912 edges in 12 s (0.53M/sec); load time: 0.18 s
2019-08-14 18:10:07  Stats for edge path 1 / 1, bucket ( 0 , 2 ): pos_rank:  6.52836 , mrr:  0.4888 , r1:  0.33052 , r10:  0.820306 , r50:  0.992694 , auc:  0.968771 , count:  6469912
2019-08-14 18:10:19  ( 0 , 3 ): Processed 6550501 e

In [83]:
!head logs/eval_train.log
print('-'*100)
!tail logs/eval_train.log

2020-10-03 20:22:18  Starting edge path 1 / 1 (./data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000_partitioned)
2020-10-03 20:22:43  ( 0 , 0 ): Processed 12966704 edges in 25 s (0.53M/sec); load time: 1.2 s
2020-10-03 20:22:43  Stats for edge path 1 / 1, bucket ( 0 , 0 ): pos_rank:  18.855 , mrr:  0.350963 , r1:  0.26817 , r10:  0.499124 , r50:  0.973969 , auc:  0.899784 , count:  12966704
2020-10-03 20:23:08  ( 0 , 1 ): Processed 12763084 edges in 24 s (0.53M/sec); load time: 0.81 s
2020-10-03 20:23:08  Stats for edge path 1 / 1, bucket ( 0 , 1 ): pos_rank:  18.649 , mrr:  0.354826 , r1:  0.272057 , r10:  0.503854 , r50:  0.976096 , auc:  0.901748 , count:  12763084
2020-10-03 20:23:34  ( 0 , 2 ): Processed 12977120 edges in 24 s (0.53M/sec); load time: 0.86 s
2020-10-03 20:23:34  Stats for edge path 1 / 1, bucket ( 0 , 2 ): pos_rank:  18.9749 , mrr:  0.349545 , r1:  0.267035 , r10:  0.497143 , r50:  0.971563 , auc:  0.898729 , count:  12977120
2020-10-03 20:2

**Result table**

| Metric   | Train | Train_new  |
|----------|-------|------------|
| pos_rank | 6.77  | 18.8       |
| MRR      | 0.48  | 0.35       |
| r1       | 0.32  | 0.27       |
| r10      | 0.81  | 0.5        |
| r50      | 0.99  | 0.97       |
| AUC      | 0.97  | 0.9        |

**Evaluating on validation**

In [5]:
!torchbiggraph_eval graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/graph-val_partitioned > logs/eval_val.log

In [67]:
!torchbiggraph_eval graph_config.py -p edge_paths={DATA_FOLDER}graph_edges/graph-val_partitioned > logs/eval_val.log

Traceback (most recent call last):
  File "/opt/conda/bin/torchbiggraph_eval", line 10, in <module>
    sys.exit(main())
  File "/opt/conda/lib/python3.7/site-packages/torchbiggraph/eval.py", line 226, in main
    do_eval(config)
  File "/opt/conda/lib/python3.7/site-packages/torchbiggraph/eval.py", line 205, in do_eval
    for _ in do_eval_and_report_stats(config, model, evaluator):
  File "/opt/conda/lib/python3.7/site-packages/torchbiggraph/eval.py", line 126, in do_eval_and_report_stats
    edge_reader = EdgeReader(edge_path)
  File "/opt/conda/lib/python3.7/site-packages/torchbiggraph/fileio.py", line 73, in __init__
    raise RuntimeError("Invalid edge dir: %s" % path)
RuntimeError: Invalid edge dir: ./data/dump/graph_edges/graph-val_partitioned


In [6]:
!head logs/eval_val.log
print('-'*100)
!tail logs/eval_val.log

2019-08-14 18:12:50  Starting edge path 1 / 1 (./data/mysql-2019-06-01/graph_edges/graph-val_partitioned)
2019-08-14 18:12:51  ( 0 , 0 ): Processed 799176 edges in 1.7 s (0.48M/sec); load time: 0.12 s
2019-08-14 18:12:51  Stats for edge path 1 / 1, bucket ( 0 , 0 ): pos_rank:  9.97506 , mrr:  0.388542 , r1:  0.236046 , r10:  0.720194 , r50:  0.97284 , auc:  0.943282 , count:  799176
2019-08-14 18:12:53  ( 0 , 1 ): Processed 822664 edges in 1.6 s (0.52M/sec); load time: 0.1 s
2019-08-14 18:12:53  Stats for edge path 1 / 1, bucket ( 0 , 1 ): pos_rank:  9.79906 , mrr:  0.39167 , r1:  0.23917 , r10:  0.723014 , r50:  0.974304 , auc:  0.944365 , count:  822664
2019-08-14 18:12:55  ( 0 , 2 ): Processed 810187 edges in 1.6 s (0.52M/sec); load time: 0.084 s
2019-08-14 18:12:55  Stats for edge path 1 / 1, bucket ( 0 , 2 ): pos_rank:  9.51358 , mrr:  0.397661 , r1:  0.243526 , r10:  0.731363 , r50:  0.975832 , auc:  0.946319 , count:  810187
2019-08-14 18:12:56  ( 0 , 3 ): Processed 819618 edges

In [68]:
!head logs/eval_val.log
print('-'*100)
!tail logs/eval_val.log

2020-10-03 04:19:54  Starting edge path 1 / 1 (./data/dump/graph_edges/graph-val_partitioned)
----------------------------------------------------------------------------------------------------
2020-10-03 04:19:54  Starting edge path 1 / 1 (./data/dump/graph_edges/graph-val_partitioned)


**Result table**

| Metric   | Train | Validation |
|----------|-------|------------|
| pos_rank | 6.77  | 10.05      |
| MRR      | 0.48  | 0.38       |
| r1       | 0.32  | 0.23       |
| r10      | 0.81  | 0.71       |
| r50      | 0.99  | 0.97       |
| AUC      | 0.97  | 0.94       |

### User evaluation

Besides that, we performed user evaluation. We engaged users that have not heard anything about our project and actively using open source projects and GitHub. As a setting for the experiment, we asked to provide five repositories that users familiar with and proposed to them two recommendations for each. First one is our best guess from Annoy nearest neighbour search and second one - random repository with 1000+ stars. Please see the table with the experiment data:

| Provided repo | Our suggestion    | Random repo                         | Choice                            |
|---------------|-------------------|-------------------------------------|-----------------------------------|
| yarnpkg/yarn  | prettier/prettier | fossasia/sphinx_fossasia_theme      | both irrelevant                   |
| npm/npm       | gulpjs/gulp       | lhartikk/ArnoldC                    | our suggestion marked as relevant |
| rspec/rspec   | rspec/rspec-core  | FLIF-hub/FLIF                       | both irrelevant                   |
| rails/rails   | sinatra/sinatra   | javamelody/javamelody               | our suggestion marked as relevant |
| ruby/ruby     | rails/rails       | ustjavac/Programming-Alpha-To-Omega | both irrelevant                   |

As we see in all cases, users correctly identified random repositories and marked 40% of our suggestions as useful, which is similar to the rank1 metric on the validation set. However, we cannot say that this result is statistically significant due to small sample size. 

Besides that, one user asked to search for repos that are similar to one particular repository for his current project task and replied that the top 10 suggestions contained "interesting results that I have not found googling".

## Prepare tensorboard

In [ ]:
# !torchbiggraph_export_to_tsv --dict {DATA_FOLDER}/graph_edges/dictionary.json \
# --checkpoint {DATA_FOLDER}/graph_edges/ --out {DATA_FOLDER}/graph_edges/embeddings.tsv

Slow ↑ method

In [86]:
import json
import h5py
import os

!mkdir -p tb
!mkdir -p tb/embeddings

def dropCategories(df, col, threshold):
    drop_list = df[col].value_counts()[threshold:].index.to_list()
    df[col].cat.remove_categories(drop_list, inplace=True)
    return df

For faster results let's see only on repos with 1K+ stars and users with 100+ followers/following

In [3]:
good_rids = pd.read_pickle('./tb/embeddings_old/repos_19k_gte1k.pkl').index.tolist()
len(good_rids)

18670

In [4]:
good_uids = pd.read_pickle('./tb/embeddings_old/users_following_29k_gte100.pkl').index.tolist()
good_uids += pd.read_pickle('./tb/embeddings_old/users_followers_24k_gte100.pkl').index.tolist()
good_uids = list(set(good_uids))
len(good_uids)

47666

Read metadata

In [89]:
!head ./data/dump/graph_edges/dictionary.json

{
    "relations": [
        "relation",
        "is_member_of",
        "follows",
        "watches"
    ],
    "entities": {
        "all": [
            "user_id_34365925",


In [100]:
rids = []
with open('./data/dump/graph_edges/dictionary.json') as f:
    d = json.load(f)
    for eid in d['entities']['all']:
        if eid.startswith('repo'):
            rids.append(int(eid.split('_')[-1]))
len(rids)

7221822

In [88]:
mc['github']['projects'].estimated_document_count()

116082837

In [7]:
repos_cur = mc['github']['projects'].find({"_id": {"$in": good_rids}}, ['url', 'language', 'created_at', 'updated_at'])
repos_df = pd.DataFrame(list(repos_cur)).set_index('_id')[['url', 'language', 'created_at', 'updated_at']]
repos_df.language = repos_df.language.astype("category")
repos_df = dropCategories(repos_df, "language", 42)
repos_df.sample(5)

,url,language,created_at,updated_at
_id,,,,
2355834,behave/behave,Python,2011-10-25 11:02:35,2019-02-25 21:52:24
9907713,gilbitron/Raneto,JavaScript,2014-05-30 10:24:18,2019-02-27 04:35:21
69826495,apollographql/apollo-server,TypeScript,2016-04-21 09:26:01,2019-02-26 15:46:21
89912090,Tencent/LKImageKit,Objective-C,2018-01-03 02:22:30,2019-02-27 06:54:36
36676484,electron/devtron,JavaScript,2016-02-12 22:57:24,2019-02-26 08:50:46


In [101]:
repos_cur = mc['github']['projects'].find({"_id": {"$in": rids[:300_000]}}, ['url', 'language', 'created_at', 'updated_at'])
repos_df = pd.DataFrame(list(repos_cur)).set_index('_id')[['url', 'language', 'created_at', 'updated_at']]
repos_df.language = repos_df.language.astype("category")
repos_df = dropCategories(repos_df, "language", 42)
repos_df.sample(5)

,url,language,created_at,updated_at
_id,,,,
35264259,yrashk/zoid,Ruby,2008-04-29 10:55:29,2019-01-09 04:14:17
68213871,hy20090501/ssm_demo001,Java,2017-06-27 06:59:20,2018-12-20 08:10:39
31978403,dineshbarai/XssScan-v1.1,NaN,2016-02-09 16:57:53,1970-01-01 01:00:01
9114419,helloyou2012/MedicalViewer,C++,2014-04-09 12:48:48,2018-12-20 05:50:22
6743671,HaeckDesign/Avedon,PHP,2013-12-15 21:43:43,2016-02-18 09:55:49


In [8]:
ufields = ['login', 'type', 'fake', 'location', 'deleted', 'country_code', 'created_at']
users_cur = mc['github']['users'].find({"_id": {"$in": good_uids}}, ufields)
users_df = pd.DataFrame(list(users_cur)).set_index('_id')[ufields]
users_df.sample(5)

,login,type,fake,location,deleted,country_code,created_at
_id,,,,,,,
333093,dolaameng,USR,False,Singapore,False,sg,2008-11-26 05:26:45
41562368,0x7214FF,USR,False,None,False,None,2011-08-31 01:31:09
569407,alekseybobkov,USR,False,None,False,None,2010-11-15 01:07:49
35798413,alexholdenmiller,USR,False,New York,False,us,2011-10-20 17:46:47
20472,dscape,USR,False,"London, United Kingdom",False,gb,2008-04-26 16:56:53


In [9]:
path = f'{DATA_FOLDER}/graph_edges'
files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.find("embeddings_all") != -1])
files = files[0:2] + files[8:] + files[2:8]
files

['./data/mysql-2019-06-01//graph_edges/embeddings_all_0.v10.h5',
 './data/mysql-2019-06-01//graph_edges/embeddings_all_1.v10.h5',
 './data/mysql-2019-06-01//graph_edges/embeddings_all_2.v10.h5',
 './data/mysql-2019-06-01//graph_edges/embeddings_all_3.v10.h5']

In [105]:
path = f'{DATA_FOLDER}graph_edges'
files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.find("embeddings_all") != -1])
# files = files[0:2] + files[8:] + files[2:8]
files

['./data/dump/graph_edges/embeddings_all_0.v10.h5',
 './data/dump/graph_edges/embeddings_all_1.v10.h5',
 './data/dump/graph_edges/embeddings_all_2.v10.h5',
 './data/dump/graph_edges/embeddings_all_3.v10.h5']

Next actions need some RAM!

In [10]:
with open(os.path.join(path, "dictionary.json"), "rt") as tf:
    dictionary = json.load(tf)
print(dictionary['relations'])
all_entities = dictionary['entities']['all']
len(all_entities)

['relation', 'watches', 'follows', 'is_member_of']


1529391

In [106]:
with open(os.path.join(path, "dictionary.json"), "rt") as tf:
    dictionary = json.load(tf)
print(dictionary['relations'])
all_entities = dictionary['entities']['all']
len(all_entities)

['relation', 'is_member_of', 'follows', 'watches']


15038812

In [11]:
pd.Series(all_entities).map(lambda x: x.split('_id_')[0]).value_counts()

user    1091930
repo     437461
dtype: int64

In [107]:
pd.Series(all_entities).map(lambda x: x.split('_id_')[0]).value_counts()

user    7816990
repo    7221822
dtype: int64

Read embeddings

In [12]:
embeddings=[]
for file in tn(files):
    with h5py.File(file, "r") as hf:
        embedding = hf["embeddings"][...]
    embeddings.append(embedding)
embeddings = np.concatenate(embeddings, axis=0)
embeddings.shape

(1529391, 100)

In [108]:
embeddings=[]
for file in tn(files):
    with h5py.File(file, "r") as hf:
        embedding = hf["embeddings"][...]
    embeddings.append(embedding)
embeddings = np.concatenate(embeddings, axis=0)
embeddings.shape

(15038812, 100)

In [13]:
emb_df = pd.Series({i: e for i, e in tn(zip(all_entities, embeddings))}).to_frame(name='embeddings')
emb_df.shape

(1529391, 1)

In [109]:
emb_df = pd.Series({i: e for i, e in tn(zip(all_entities, embeddings))}).to_frame(name='embeddings')
emb_df.shape

(15038812, 1)

Combine all metainfo with embeddings

In [14]:
repos_emb_df = pd.Series({int(i.split('_')[-1]): e for i, e in emb_df.loc[[f'repo_id_{i}' for i in repos_df.index]]['embeddings'].iteritems()}).to_frame(name='embeddings')
repos_df = repos_df.join(repos_emb_df)
repos_df.sample(5)

,url,language,created_at,updated_at,embeddings
_id,,,,,
14471725,justjavac/flarum,JavaScript,2014-12-24 14:01:38,2019-02-27 05:07:21,"[0.20021674, -0.035547603, 0.14687921, 0.09368..."
266651,joewalnes/reconnecting-websocket,JavaScript,2012-01-27 20:28:51,2019-02-27 07:47:27,"[0.8540273, 0.00038430316, 0.2725552, 0.238947..."
28281221,up-for-grabs/up-for-grabs.net,JavaScript,2013-11-20 22:44:13,2019-02-26 12:30:42,"[0.13455452, -0.08811931, -0.020408232, 0.1868..."
5828347,HabitRPG/habitrpg,JavaScript,2012-06-06 22:49:48,2016-02-18 09:17:21,"[0.6335141, -0.049198274, -0.040939253, 0.0998..."
32907987,pybind/pybind11,C++,2015-07-05 19:46:48,2019-02-26 10:47:07,"[0.31675598, -0.11396529, 0.041127887, -0.0468..."


In [110]:
repos_emb_df = pd.Series({int(i.split('_')[-1]): e for i, e in emb_df.loc[[f'repo_id_{i}' for i in repos_df.index]]['embeddings'].iteritems()}).to_frame(name='embeddings')
repos_df = repos_df.join(repos_emb_df)
repos_df.sample(5)

,url,language,created_at,updated_at,embeddings
_id,,,,,
35203230,Abastro/StellarAPI,Java,2016-04-16 08:01:46,1970-01-01 01:00:01,"[-0.24669535, -0.13281979, -0.39560172, 0.1111..."
14337770,kocourmatej/FinalLittlenightmares,NaN,2014-12-22 15:17:51,2016-02-24 20:11:23,"[-0.021142844, 0.060435604, 0.07376396, 0.4121..."
1488984,prapin/LuaBrainFuck,Lua,2012-11-14 21:53:41,2016-11-29 13:54:04,"[-0.2751308, 0.084774256, -0.603459, -0.078731..."
19669902,superzhangby/starbuck,Java,2015-05-21 20:31:31,2016-03-04 16:53:31,"[-0.036725752, 0.19546543, -0.07749834, 0.4740..."
6942260,almonk/twoup,Objective-C,2013-12-15 20:56:13,2016-02-04 22:14:51,"[0.12220988, 0.13873668, 0.2711601, -0.1434591..."


In [15]:
users_emb_df = pd.Series({int(i.split('_')[-1]): e for i, e in emb_df.loc[[f'user_id_{i}' for i in users_df.index]]['embeddings'].iteritems()}).to_frame(name='embeddings')
users_df = users_df.join(users_emb_df)
users_df.sample(5)

,login,type,fake,location,deleted,country_code,created_at,embeddings
_id,,,,,,,,
12539209,NathAston,USR,False,London,False,gb,2016-03-14 11:17:47,"[-0.43741047, -0.10959671, 0.17184402, 0.16799..."
14376,jsjohnst,USR,False,New York City,False,us,2008-04-07 02:20:43,"[-0.77265024, 0.03190932, 0.13513942, -0.00035..."
31888096,AeronStory,USR,False,None,True,None,2016-10-18 15:04:57,"[-0.97922194, 0.38436, 0.2280939, 0.09458058, ..."
39277352,higgsfield,USR,False,None,False,None,2016-12-06 23:12:52,"[0.5044715, -0.032197986, 0.0016144309, 0.0136..."
13376482,Ljzn,USR,False,home,False,us,2016-05-20 16:57:18,"[-0.77141976, 0.25560012, -0.005199214, -0.012..."


Save for TensorBoard

In [52]:
repos_df.to_pickle("./tb/embeddings/repos_19k_gte1k.pkl")

In [30]:
users_df.to_pickle("./tb/embeddings/users_48k_gte100.pkl")

If trained with `global = True` - embeddings for different types could be comparable (same hyperspace)

In [ ]:
# repos_df.index = repos_df.index.map(lambda x: f'r{x}')
# users_df.index = users_df.index.map(lambda x: f'u{x}')
# combined_df = pd.concat([repos_df, users_df])
# combined_df.to_pickle("./tb/embeddings/repos_19k_gte1k_users_48k_gte100.pkl")

↑ embeddings for repos and users are far away from each other - no sense in viewing into combined multidimentional space

(reason - no `global = True` in config for better independent representations)

In [ ]:
!rm -rf ./tb/embeddings/.ipynb_checkpoints

Now, `cd tb` & `run2.sh` at terminal with pre-installed docker or read `tb/README.md`

Then, open `HOSTNAME:PORT`, where `HOSTNAME` cane from machine, where the last script ran and `PORT` = `8002` (by default, but configurable)

## Paper2code DB

In [197]:
!head ./data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000.csv

from_entity_id	relation	to_entity_id
user_id_1	is_member_of	repo_id_1
user_id_2	is_member_of	repo_id_1
user_id_4	is_member_of	repo_id_1
user_id_24	is_member_of	repo_id_3
user_id_5465	is_member_of	repo_id_3
user_id_11834	is_member_of	repo_id_3
user_id_11983	is_member_of	repo_id_3
user_id_17241	is_member_of	repo_id_3
user_id_26333	is_member_of	repo_id_3


In [199]:
arr = []
with open('./data/dump/graph_edges/part-00000-292466f2-30c4-4402-b783-2ec7738d245a-c000.csv') as f:
    for line in tn(f):
        arr.append(line.split('\t')[1])
Counter(arr)

Counter({'relation': 1,
         'is_member_of': 14129288,
         'follows': 29809738,
         'watches': 176154850})

In [201]:
carr = []
for uw in tn(mc['github']['users_watches'].find({}, {'_id': -1, 'repos_count': 1})):
    carr.append(uw['repos_count'])
len(carr), pd.Series(carr).describe()

(6012456, count    6.012456e+06
 mean     2.307706e+01
 std      1.647244e+02
 min      1.000000e+00
 25%      1.000000e+00
 50%      2.000000e+00
 75%      9.000000e+00
 max      2.252900e+05
 dtype: float64)

In [202]:
sum(carr)

138749808

In [145]:
!tar -C ./data -xvzf ./data/github_stargazers_graph.json.tar.gz

github_stargazers_graph.json


In [246]:
!tar -C ./data -xvzf ./data/github_stargazers_graph_detailled_v5.json.tar.gz

github_stargazers_graph_detailled_v5.json


In [155]:
sg_df = []
with open('./data/github_stargazers_graph.json') as f:
    for line in f:
        sg_df.append(json.loads(line.strip('\n')))
sg_df = pd.DataFrame(sg_df)
sg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62633 entries, 0 to 62632
Data columns (total 6 columns):
InternalPaperId       62633 non-null int64
InternalRepoId        62633 non-null int64
RemoteRepoFullName    62633 non-null object
RemoteRepoStarred     62633 non-null int64
RemoteRepoUrl         62633 non-null object
RemoteStargazers      52966 non-null object
dtypes: int64(3), object(3)
memory usage: 2.9+ MB


In [247]:
sg_df = []
with open('./data/github_stargazers_graph_detailled_v5.json') as f:
    for line in f:
        sg_df.append(json.loads(line.strip('\n')))
sg_df = pd.DataFrame(sg_df)
sg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49398 entries, 0 to 49397
Data columns (total 13 columns):
InternalPaperId                   49398 non-null int64
InternalRepoId                    49398 non-null int64
RemoteForks                       33531 non-null object
RemoteReadmeRelationshipArxiv     43917 non-null object
RemoteReadmeRelationshipGithub    35090 non-null object
RemoteRepoFullName                49398 non-null object
RemoteRepoId                      49398 non-null int64
RemoteRepoLastUpdate              49398 non-null object
RemoteRepoStarred                 49398 non-null int64
RemoteRepoUrl                     49398 non-null object
RemoteStargazers                  39918 non-null object
RemoteTopics                      9996 non-null object
RemoteWatchers                    49331 non-null object
dtypes: int64(4), object(9)
memory usage: 4.9+ MB


In [248]:
sg_df

,InternalPaperId,InternalRepoId,RemoteForks,RemoteReadmeRelationshipArxiv,RemoteReadmeRelationshipGithub,RemoteRepoFullName,RemoteRepoId,RemoteRepoLastUpdate,RemoteRepoStarred,RemoteRepoUrl,RemoteStargazers,RemoteTopics,RemoteWatchers
0,9,1723826,"[{'ForkUserId': 37745685, 'ForkUserLogin': 'Ja...",[https://arxiv.org/abs/1703.09471],"[https://github.com/drvinceknight/Nashpy, http...",coallaoh/AIP,88062408,2020-05-29T23:18:55Z,11,https://github.com/coallaoh/AIP,"[{'StargazerRemoteId': 7447092, 'StargazerLogi...",None,"[{'WatcherRemoteId': 7447092, 'WatcherLogin': ..."
1,10,1538136,"[{'ForkUserId': 6200208, 'ForkUserLogin': 'KSr...",[https://arxiv.org/pdf/2006.12122.pdf],[https://github.com/facebookresearch/torchbeas...,facebookresearch/impact-driven-exploration,235892749,2020-07-08T07:48:58Z,44,https://github.com/facebookresearch/impact-dri...,"[{'StargazerRemoteId': 12811398, 'StargazerLog...",None,"[{'WatcherRemoteId': 5103945, 'WatcherLogin': ..."
2,16,1538134,"[{'ForkUserId': 59344822, 'ForkUserLogin': 'An...",None,"[https://github.com/LIBOL/SOL, https://github....",LIBOL/SOL,63072935,2020-07-08T07:48:58Z,87,https://github.com/LIBOL/SOL,"[{'StargazerRemoteId': 10544043, 'StargazerLog...",None,"[{'WatcherRemoteId': 6936608, 'WatcherLogin': ..."
3,18,35380,"[{'ForkUserId': 24252517, 'ForkUserLogin': 'dm...",[https://arxiv.org/abs/1707.06226],None,debanjanghosh/sarcasm_context,95561750,2020-04-17T07:57:11Z,6,https://github.com/debanjanghosh/sarcasm_context,"[{'StargazerRemoteId': 128885, 'StargazerLogin...",None,"[{'WatcherRemoteId': 5402638, 'WatcherLogin': ..."
4,18,1538137,"[{'ForkUserId': 24252517, 'ForkUserLogin': 'dm...",[https://arxiv.org/pdf/1707.06226.pdf],None,Alex-Fabbri/deep_learning_nlp_sarcasm,82968519,2020-07-08T07:48:58Z,7,https://github.com/Alex-Fabbri/deep_learning_n...,"[{'StargazerRemoteId': 21278781, 'StargazerLog...",None,"[{'WatcherRemoteId': 6563196, 'WatcherLogin': ..."
5,23,1538133,None,None,[https://github.com/sinaahmadi/wergor],sinaahmadi/wergor,99863744,2020-07-08T07:48:58Z,3,https://github.com/sinaahmadi/wergor,"[{'StargazerRemoteId': 18519747, 'StargazerLog...",None,"[{'WatcherRemoteId': 12765529, 'WatcherLogin':..."
6,23,3957966,None,[https://arxiv.org/abs/1811],None,459548764/Pairwise-Indepence-Autoencoder,194267321,2020-03-26T05:17:45Z,5,https://github.com/459548764/Pairwise-Indepenc...,None,None,"[{'WatcherRemoteId': 69664243, 'WatcherLogin':..."
7,29,2784854,None,"[http://arxiv.org/abs/1711.08998, https://arxi...","[https://github.com/baumgach/vagan-code, https...",amirunpri2018/Visual-Feature-Attribution-Using...,168823076,2020-06-10T07:30:45Z,2,https://github.com/amirunpri2018/Visual-Featur...,None,None,"[{'WatcherRemoteId': 45308941, 'WatcherLogin':..."
8,35,1538141,"[{'ForkUserId': 6565882, 'ForkUserLogin': 'gis...",None,"[https://github.com/charlesq34/pointnet2, http...",WeikaiTan/Toronto-3D,249069745,2020-07-08T07:48:58Z,25,https://github.com/WeikaiTan/Toronto-3D,"[{'StargazerRemoteId': 59512663, 'StargazerLog...",None,"[{'WatcherRemoteId': 13037776, 'WatcherLogin':..."
9,38,1538140,"[{'ForkUserId': 17541316, 'ForkUserLogin': 'ML...",[https://arxiv.org/pdf/1806.08804.pdf],None,RexYing/diffpool,124960476,2020-07-08T07:48:58Z,270,https://github.com/RexYing/diffpool,"[{'StargazerRemoteId': 6671595, 'StargazerLogi...",None,"[{'WatcherRemoteId': 2379898, 'WatcherLogin': ..."


In [179]:
repo_query = 'graphx'

sg_df[sg_df['RemoteRepoFullName'].str.lower().str.contains(repo_query)]

,InternalPaperId,InternalRepoId,RemoteRepoFullName,RemoteRepoStarred,RemoteRepoUrl,RemoteStargazers
27562,119424,664365,justanhduc/graphx-conv,32,https://github.com/justanhduc/graphx-conv,"[{'StargazerRemoteId': 31762972, 'StargazerLog..."
27563,119424,1544807,justanhduc/graphx-conv,31,https://github.com/justanhduc/graphx-conv,"[{'StargazerRemoteId': 31762972, 'StargazerLog..."


In [181]:
repo_query = 'declare-lab/awesome-sentiment-analysis'

sg_df[sg_df['RemoteRepoFullName'].str.lower().str.contains(repo_query)]

,InternalPaperId,InternalRepoId,RemoteRepoFullName,RemoteRepoStarred,RemoteRepoUrl,RemoteStargazers
1793,7883,1539665,declare-lab/awesome-sentiment-analysis,164,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
36915,157653,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
56890,195687,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
57131,200513,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
57442,207782,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
57630,210997,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
57831,215581,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
57910,218089,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."
57960,219791,201,declare-lab/awesome-sentiment-analysis,165,https://github.com/declare-lab/awesome-sentime...,"[{'StargazerRemoteId': 13835142, 'StargazerLog..."


In [178]:
sg_df['RemoteStargazers'].dropna().map(len).describe()

count    52966.000000
mean       167.438545
std        997.904983
min          1.000000
25%          5.000000
50%         18.000000
75%         73.000000
max      40000.000000
Name: RemoteStargazers, dtype: float64

In [173]:
for repo_query in sg_df['RemoteRepoFullName']:
    repo_query = repo_query.lower()
    if not repos_df[repos_df['url'].str.lower().str.contains(repo_query)].empty:
        print(repo_query)

libol/sol
insomnia250/channel-prune


KeyboardInterrupt: 

In [174]:
repo_query = 'libol/sol'

repos_df[repos_df['url'].str.lower().str.contains(repo_query)]

,url,language,created_at,updated_at,embeddings
_id,,,,,
44371847,LIBOL/SOL,NaN,2016-07-11 14:03:08,2019-02-25 07:49:49,"[0.09391572, 0.53017694, -0.44245782, 0.257839..."


In [188]:
len(sg_df['RemoteRepoFullName'].unique().tolist())

49498

In [194]:
sg_df.sample(1)['RemoteStargazers'].values[0]

[{'StargazerRemoteId': 28194863, 'StargazerLogin': 'SongFGH'},
 {'StargazerRemoteId': 8077469, 'StargazerLogin': 'andreasasprou'},
 {'StargazerRemoteId': 18051187, 'StargazerLogin': 'loveJasmine'},
 {'StargazerRemoteId': 7035357, 'StargazerLogin': 'yanzhang92'},
 {'StargazerRemoteId': 14815963, 'StargazerLogin': 'weiyudang'},
 {'StargazerRemoteId': 315967, 'StargazerLogin': 'mrjj'},
 {'StargazerRemoteId': 39670256, 'StargazerLogin': 'sw-gong'},
 {'StargazerRemoteId': 24446986, 'StargazerLogin': 'cristianoBY'},
 {'StargazerRemoteId': 18097297, 'StargazerLogin': 'Wangyang123wy'},
 {'StargazerRemoteId': 699239, 'StargazerLogin': 'treper'},
 {'StargazerRemoteId': 7530936, 'StargazerLogin': 'liuzhenqi77'},
 {'StargazerRemoteId': 21268629, 'StargazerLogin': 'lexparsimon'},
 {'StargazerRemoteId': 18417945, 'StargazerLogin': 'dterg'},
 {'StargazerRemoteId': 44162052, 'StargazerLogin': 'captify-dieter'},
 {'StargazerRemoteId': 7662185, 'StargazerLogin': 'MohammadHeydari'},
 {'StargazerRemoteId'

In [196]:
!head ./data/dump/watchers.csv

1,1,"2009-12-08 10:17:27"
1,2,"2009-12-08 10:17:27"
1,4,"2009-12-08 10:17:27"
1,6,"2010-02-05 06:35:04"
1,7,"2009-12-08 10:17:27"
1,8,"2010-03-26 04:57:36"
1,9,"2009-12-08 10:17:27"
1,10,"2009-12-08 10:17:27"
1,11,"2009-12-08 10:17:27"
1,12,"2009-12-08 10:17:27"


In [186]:
mc['github']['projects'].count_documents({"url": {"$in": sg_df['RemoteRepoFullName'].tolist()}})

23819

## Building Annoy (library to do nearest neighbor search)

In [1]:
from annoy import AnnoyIndex

### Repos

Read dataframe

In [8]:
repos_df = pd.read_pickle("./tb/embeddings/repos_19k_gte1k.pkl")
repos_df.shape

(18670, 5)

Dimension

In [225]:
dim = len(repos_df['embeddings'].iloc[0])
dim

100

Mapping for index

In [10]:
repos_ann = AnnoyIndex(dim, 'angular')  # Length of item vector that will be indexed
repos_mapping = {}
for i, (repo_id, e) in enumerate(repos_df['embeddings'].iteritems()):
    repos_ann.add_item(i, list(e))
    repos_mapping[repo_id] = i

Build index

In [11]:
n_trees = 1_000

%time repos_ann.build(n_trees)

CPU times: user 38.7 s, sys: 25.3 ms, total: 38.8 s
Wall time: 38.8 s


True

In [114]:
n_trees = 1_000

%time repos_ann.build(n_trees)

CPU times: user 13min 50s, sys: 2.61 ms, total: 13min 50s
Wall time: 13min 50s


True

Save index

In [12]:
repos_ann.save('./data/repos.ann')
pd.to_pickle(repos_mapping, './data/repos_mapping.pkl')

### Users

Read dataframe

In [13]:
users_df = pd.read_pickle("./tb/embeddings/users_48k_gte100.pkl")
users_df.shape

(47666, 8)

Dimension

In [14]:
dim = len(users_df['embeddings'].iloc[0])
dim

100

Mapping for index

In [15]:
users_ann = AnnoyIndex(dim, 'angular')
users_mapping = {}
for i, (user_id, e) in enumerate(users_df['embeddings'].iteritems()):
    users_ann.add_item(i, list(e))
    users_mapping[user_id] = i

Build index

In [16]:
n_trees = 1_000

%time users_ann.build(n_trees)

CPU times: user 1min 54s, sys: 147 ms, total: 1min 54s
Wall time: 1min 54s


True

Save index

In [17]:
users_ann.save('./data/users.ann')
pd.to_pickle(users_mapping, './data/users_mapping.pkl')

## Recommender search example

### Repos

Read dataframe

In [2]:
repos_df = pd.read_pickle("./tb/embeddings/repos_19k_gte1k.pkl")
repos_df.shape

(18670, 5)

Dimension

In [3]:
dim = len(repos_df['embeddings'].iloc[0])
dim

100

Read index & mapping

In [4]:
repos_ann = AnnoyIndex(dim, 'angular')
repos_ann.load('./data/repos.ann') # super fast, will just mmap the file
repos_ann.get_n_items(), repos_ann.get_n_trees()

(18670, 1000)

In [5]:
repos_mapping = pd.read_pickle('./data/repos_mapping.pkl')
repos_mapping_rev = {v: k for k, v in repos_mapping.items()}
len(repos_mapping)

18670

In [6]:
repos_mapping_rev = {v: k for k, v in repos_mapping.items()}
len(repos_mapping_rev)

18670

In [7]:
def get_nns_by_repo_id(repo_id, n_nn=10):
    res, dist = repos_ann.get_nns_by_item(repos_mapping[repo_id], n_nn, include_distances=True) # will find the n nearest neighbors
    res = [repos_mapping_rev[r] for r in res]
    res = repos_df.loc[res]
    res['distance'] = [round(d, 3) for d in dist]
    return res

Type interesting query for repos search (don't forget about unpopularity limitation)

In [8]:
repo_query = 'apache/spark'

repos_df[repos_df['url'].str.lower().str.contains(repo_query)]

,url,language,created_at,updated_at,embeddings
_id,,,,,
8196280,apache/spark,Scala,2014-02-25 08:00:08,2019-02-26 23:57:18,"[0.5067623, 0.013571289, -0.07234435, -0.00304..."


In [9]:
repo_query = 'graphx'

repos_df[repos_df['url'].str.lower().str.contains(repo_query)]

,url,language,created_at,updated_at,embeddings
_id,,,,,


Choose needed one to search nearest neighbors

In [10]:
repo_id = 8196280

get_nns_by_repo_id(repo_id)

,url,language,created_at,updated_at,embeddings,distance
_id,,,,,,
8196280,apache/spark,Scala,2014-02-25 08:00:08,2019-02-26 23:57:18,"[0.5067623, 0.013571289, -0.07234435, -0.00304...",0.000
45297,apache/kafka,Java,2011-08-15 18:06:16,2019-02-27 04:05:00,"[0.69850904, 0.042590182, 0.0013614872, -0.006...",0.422
11610797,apache/storm,Java,2013-11-05 08:00:14,2019-02-27 04:45:34,"[0.59941286, 0.078122355, 0.035178997, -0.0098...",0.445
14807988,apache/flink,Java,2014-06-07 07:00:10,2019-02-27 04:11:52,"[0.4457919, 0.11843297, -0.097657256, 0.076890...",0.460
39068,apache/hadoop,Java,2009-03-27 14:41:53,2019-02-26 11:50:35,"[0.45037118, 0.10151473, 0.007142633, 0.081586...",0.491
13773,apache/hbase,Java,2014-05-23 07:00:07,2019-02-27 00:06:19,"[0.58324295, 0.006161812, -0.060786575, -0.005...",0.522
10305482,databricks/learning-spark,Java,2014-06-16 04:47:54,2019-02-26 10:02:28,"[0.27490905, -0.076953545, -0.19611663, -0.061...",0.548
1366147,neo4j/neo4j,Java,2012-11-12 08:46:15,2019-02-27 02:33:20,"[0.46941158, -0.06264764, 0.00042413553, -0.08...",0.564
10045,akka/akka,Scala,2009-02-16 12:51:54,2019-02-26 10:48:39,"[0.7051094, -0.10073524, 0.014133396, -0.09673...",0.566


All Big Data, distributed tools

In [134]:
repo_id = 5394634

get_nns_by_repo_id(repo_id)

,url,language,created_at,updated_at,embeddings,distance
_id,,,,,,
5394634,amplab/graphx,Scala,2013-09-17 21:11:32,2019-02-27 05:07:32,"[-0.010923385, 1.0827434, 0.0888906, 0.1877617...",0.000
13494877,databricks/spark-avro,Scala,2014-09-30 17:50:58,2019-02-25 20:10:56,"[0.07442512, 0.93862367, 0.25723663, 0.0609760...",0.578
13685290,h2oai/sparkling-water,Scala,2014-10-13 23:06:35,2019-02-26 09:51:27,"[0.06469482, 1.099439, -0.3481848, 0.23413944,...",0.623
22102471,Huawei-Spark/Spark-SQL-on-HBase,Scala,2015-07-15 21:52:43,2019-02-25 07:12:59,"[0.01358583, 0.6700988, 0.071986, 0.043676954,...",0.631
4192364,Netflix/genie,Java,2013-06-20 20:35:56,2019-02-27 08:42:10,"[0.106297165, 0.8218349, 0.07169129, 0.3951361...",0.650
11537020,spark-jobserver/spark-jobserver,Scala,2014-08-21 23:07:47,2019-02-26 08:57:58,"[0.1536507, 0.61761206, 0.25017357, 0.07854915...",0.651
10193032,MicrosoftResearch/Dryad,C#,2013-07-17 22:43:10,2019-01-27 09:38:07,"[-0.14482208, 1.1273171, -0.5983114, -0.043837...",0.659
145486,apache/hama,Java,2010-08-25 07:00:06,2019-01-11 22:02:08,"[-0.38115755, 1.0068301, 0.08077888, 0.1294572...",0.669
77042,nathanmarz/elephantdb,Java,2011-02-16 01:54:18,2019-02-01 01:51:34,"[-0.023104027, 0.7960348, 0.41653335, 0.166897...",0.682


### Users

Read dataframe

In [10]:
users_df = pd.read_pickle("./tb/embeddings/users_48k_gte100.pkl")
users_df.shape

(47666, 8)

Dimension

In [11]:
dim = len(users_df['embeddings'].iloc[0])
dim

100

Read index & mapping

In [12]:
users_ann = AnnoyIndex(dim, 'angular')
users_ann.load('./data/users.ann') # super fast, will just mmap the file
users_mapping = pd.read_pickle('./data/users_mapping.pkl')
users_mapping_rev = {v: k for k, v in users_mapping.items()}
len(users_mapping)

47666

In [13]:
def get_nns_by_user_id(user_id, n_nn=10):
    res, dist = users_ann.get_nns_by_item(users_mapping[user_id], n_nn, include_distances=True) # will find the n nearest neighbors
    res = [users_mapping_rev[r] for r in res]
    res = users_df.loc[res]
    res['distance'] = [round(d, 3) for d in dist]
    return res

Type interesting query for users search (don't forget about unpopularity limitation)

In [14]:
user_query = 'wiki'

users_df[users_df['login'].str.lower().str.contains(user_query)]

,login,type,fake,location,deleted,country_code,created_at,embeddings
_id,,,,,,,,
1652246,wikibook,USR,False,None,False,None,2013-02-22 06:53:02,"[-0.12921746, -0.4415784, -0.40390605, 0.12089..."
4016654,wikimatze,USR,False,Berlin,False,de,2010-05-04 14:46:36,"[-0.3107245, 0.10202297, 0.2773474, -0.2349214..."


Choose needed one to search nearest neighbors

In [15]:
user_id = 1652246

get_nns_by_user_id(user_id)

,login,type,fake,location,deleted,country_code,created_at,embeddings,distance
_id,,,,,,,,,
1652246,wikibook,USR,False,None,False,None,2013-02-22 06:53:02,"[-0.12921746, -0.4415784, -0.40390605, 0.12089...",0.000
3617048,gilbutITbook,USR,False,서울특별시 마포구 서교동 467-9,False,None,2014-03-19 05:48:20,"[0.1376717, -0.29175922, -0.17365614, -0.05276...",0.839
286093,javajigi,USR,False,None,False,None,2010-12-13 00:06:41,"[0.064416006, -0.28612432, -0.28292352, 0.0340...",0.853
182762,dalinaum,USR,False,The Peach Blossom Spring,False,None,2009-10-28 01:13:40,"[-0.86923313, -0.008363037, -0.31862542, 0.166...",0.873
884777,ihoneymon,USR,False,"Seoul, South Korea",False,kr,2011-06-26 12:56:10,"[-0.037841484, -0.36508644, -0.052486323, 0.08...",0.894
5712011,arahansa,USR,False,"Incheon, South Korea",False,None,2014-01-18 12:54:37,"[-0.5806951, -0.10983159, 0.040602323, -0.3568...",0.909
13295853,seoul-opengov,USR,False,"seoul, korea",False,kr,2016-04-25 09:22:59,"[-0.26374817, -0.39799082, -0.021314356, -0.03...",0.912
68757,jongman,USR,False,None,False,None,2010-03-22 05:25:55,"[-0.29828942, -0.373523, 0.040735707, 0.066990...",0.915
68445,Sangwook,USR,False,seoul,False,kr,2009-09-05 05:40:53,"[-0.6826483, -0.31995863, -0.1767753, 0.093110...",0.937


Korean "cluster"

If we go for their profiles - some similar stuff could be found there

It is essential to remember about absolute distance, not just TOP N, because some entities have neighbors close to them, while others could have not

## Flask API

In [243]:
repos_df = pd.read_pickle("./tb/embeddings_old/repos_85k_gte200.pkl")
repos_df.shape

(84806, 6)

In [244]:
repos_df

,url,language,created_at,updated_at,n_vecs,embeddings
_id,,,,,,
3,matplotlib/basemap,C++,2011-02-19 02:58:42,2019-02-20 11:59:42,473,"[0.01573314517736435, 0.017966514453291893, 0...."
6,cocos2d/cocos2d-x,C++,2010-11-18 23:17:00,2019-02-26 20:51:59,13901,"[0.24193669855594635, 0.1203954815864563, 0.49..."
19,samuelclay/NewsBlur,Objective-C,2009-01-05 14:00:43,2019-02-26 03:37:41,5455,"[0.1151142343878746, 0.117125004529953, 0.4360..."
23,adammark/Markup.js,JavaScript,2011-08-23 02:30:04,2019-02-11 19:14:24,292,"[0.05879637226462364, -0.04485967010259628, 0...."
28,luislavena/rake-compiler,Ruby,2008-11-03 11:19:43,1970-01-01 01:00:01,252,"[-0.11316311359405518, -0.4010695815086365, 0...."
29,opencv/opencv,C++,2012-07-19 09:40:17,2016-10-11 14:11:27,7744,"[0.17094147205352783, -0.06545144319534302, 0...."
31,gpjt/webgl-lessons,Python,2009-10-06 12:30:19,2019-02-21 16:17:59,597,"[0.02872665412724018, -0.034932736307382584, 0..."
34,pockethub/PocketHub,JavaScript,2011-09-08 16:52:50,2016-10-11 14:11:27,5588,"[0.22444790601730347, -0.04315068945288658, 0...."
37,angular/angular.js,JavaScript,2010-01-06 00:34:37,2019-02-27 03:14:26,56025,"[0.23754794895648956, -0.08706655353307724, 0...."


In [245]:
len(set(repos_df['url']) & p2c_repo_urls)

1722

In [241]:
p2c_repo_urls = set(sg_df['RemoteRepoFullName'].tolist())
len(p2c_repo_urls)

49498

[CHOSEN_DOMAIN]/api/v1.0/get_closest_repos?gh_url=[GH_REPO_URL]&top_n=10&
